In [ ]:
!python /content/drive/MyDrive/flipkart/m1_data_ingestion.py

[M1] Loaded raw dataset: 8173 rows, 46 columns
[M1] Parsed 6 timestamp columns
[M1] Dropped 149 invalid rows (8173 -> 8024)
[M1] Valid (plausible, in-bbox) stretch values: 606 / 8024 rows (7.6%)
[M1] Top 20 corridors by incident count:
      Non-corridor: 3051 incidents
      Mysore Road: 728 incidents
      Bellary Road 1: 607 incidents
      Tumkur Road: 458 incidents
      Bellary Road 2: 379 incidents
      Hosur Road: 297 incidents
      ORR North 1: 274 incidents
      Old Madras Road: 257 incidents
      Magadi Road: 243 incidents
      ORR East 1: 242 incidents
      ORR North 2: 235 incidents
      Bannerghata Road: 208 incidents
      ORR East 2: 183 incidents
      West of Chord Road: 172 incidents
      ORR West 1: 166 incidents
      CBD 2: 98 incidents
      IRR(Thanisandra road): 95 incidents
      Hennur Main Road: 94 incidents
      Varthur Road: 75 incidents
      Old Airport Road: 74 incidents

[M1] Saved clean dataset -> /content/drive/MyDrive/flipkart/clean_inciden

In [ ]:
!python /content/drive/MyDrive/flipkart/m2_feature_engineering.py

[M2] Loaded clean dataset: 8024 rows
[M2] Training corridor models on 4973 labeled rows (21 unique corridors)
[M2] RF corridor model training accuracy: 1.0000
[M2] KNN corridor model training accuracy: 1.0000
[M2] Saved corridor models (RF + KNN) -> /content/drive/MyDrive/flipkart/corridor_model.pkl
[M2] Rows needing corridor prediction: 3051 / 8024
[M2] Predictions accepted (RF/KNN agree AND combined conf >= 0.8): 1208 / 3051 (39.6%)
[M2] RF/KNN agreement rate on predicted rows: 0.6899
[M2] corridor_source breakdown:
corridor_source
original      4973
unresolved    1843
predicted     1208
[M2] Built 5 CIS signal tables (using corridor_final for w_corridor):
      w_cause      : 16 causes
      w_planned    : 2 event types
      w_escalation : 2 states
      w_hour       : 24 hours
      w_corridor   : 22 corridors

[M2] signal_score stats:
count    8024.000000
mean        0.074153
std         0.022927
min         0.040552
25%         0.061062
50%         0.067451
75%         0.075546


In [ ]:
!python /content/drive/MyDrive/flipkart/m3_tgcf_calibration.py

[M3] Loaded existing cache: 273 cached responses (already-paid-for calls will be skipped)
[M3] Purged 0 stale fallback entries from cache (273 valid entries kept). Safe to re-run now.
[M3] 21 real corridors for MapMyIndia calibration:
      Mysore Road: 925 incidents
      Bellary Road 1: 683 incidents
      Tumkur Road: 560 incidents
      Bellary Road 2: 423 incidents
      ORR East 2: 405 incidents
      Hosur Road: 361 incidents
      Magadi Road: 343 incidents
      ORR West 1: 339 incidents
      ORR North 1: 284 incidents
      ORR North 2: 280 incidents
      ORR East 1: 275 incidents
      Old Madras Road: 273 incidents
      Bannerghata Road: 225 incidents
      West of Chord Road: 193 incidents
      Hennur Main Road: 118 incidents
      IRR(Thanisandra road): 113 incidents
      CBD 2: 106 incidents
      Airport New South Road: 97 incidents
      Varthur Road: 77 incidents
      Old Airport Road: 75 incidents
      CBD 1: 26 incidents
[M3] Real hour is 14:00 -> filling TIM

In [ ]:
!python /content/drive/MyDrive/flipkart/m4a_closure_classifier.py

[M4a] Road closure model - improved blend using M2 corridor_final
[M4a] Loaded raw data: 8173 rows from /content/drive/MyDrive/flipkart/data.csv
[M4a] Loaded M2 corridor model bundle -> /content/drive/MyDrive/flipkart/corridor_model.pkl
[M4a] Dataset: 8173 rows | closure rate: 8.27%
[M4a] Selected threshold: 0.230
[M4a] Test precision=0.4702 recall=0.5259 f1=0.4965 roc_auc=0.8188
[M4a] Saved closure bundle -> /content/drive/MyDrive/flipkart/closure_model_bundle.pkl
[M4a] Saved metrics -> /content/drive/MyDrive/flipkart/closure_eval_metrics.json


In [ ]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.7 MB/s eta 0:00:00


In [ ]:
!nohup streamlit /content/drive/MyDrive/flipkart/streamlit_app.py --server.port 8501 --server.headless true > log.txt 2>&1 &

In [ ]:
!streamlit run /content/drive/MyDrive/flipkart/streamlit_app.py --server.port 8501 &



2026-06-22 17:48:57.519 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.87.53.132:8501

  Stopping...


In [ ]:
!python /content/drive/MyDrive/flipkart/m4b_congestion_scorer.py

[M4b] Loaded feature matrix: 8024 rows
[M4b] Loaded ETA baselines for 21 corridors (excluding _meta)
[M4b] Trained nearest-neighbor corridor lookup on 6181 labeled points
[M4b] Assigned nearest-neighbor baseline corridor to 1843 Non-corridor rows
[M4b] Nearest-neighbor distance stats (km): mean=1.317, median=0.751, max=20.439
[M4b] WARNING: 335 rows have nearest-neighbor distance > 2km (weak match - baseline borrowed from a fairly distant corridor)
[M4b] max_possible (95th percentile of raw_score): 13.6336

[M4b] CIS distribution:
count    8024.000000
mean        1.605994
std         2.760051
min         0.000000
25%         0.000000
50%         0.000000
75%         2.126090
max        10.000000
Name: CIS, dtype: float64

[M4b] === CIS validation against real closure outcome ===
      Mean CIS (closure=True)  : 1.965
      Mean CIS (closure=False) : 1.577
      Difference                : +0.387
      (Honest framing: this is a sanity check, not proof of true severity -
       no conti

In [ ]:
!python /content/drive/MyDrive/flipkart/m4c_incident_forecaster.py

[M4c] Loaded 8024 incidents with corridor_final + start_datetime
[M4c] Date range: 2023-11-09 19:24:48.154000+00:00 -> 2024-04-08 17:11:42.780000+00:00

[M4c] Forecasting 22 corridor series (including 'Non-corridor' as its own series, per locked decision)
[M4c] Train period: start of data -> 2024-02-29
[M4c] Test period : 2024-03-01 -> end of data

[M4c] === Confidence assessment per corridor ===
[M4c]   Airport New South Road    avg/wk=  4.22  zero_weeks= 4/23  nonzero_weeks=19  -> LOW_CONFIDENCE (avg_weekly=4.22 < 8.0)
[M4c]   Bannerghata Road          avg/wk=  9.78  zero_weeks= 1/23  nonzero_weeks=22  -> OK
[M4c]   Bellary Road 1            avg/wk= 29.70  zero_weeks= 0/23  nonzero_weeks=23  -> OK
[M4c]   Bellary Road 2            avg/wk= 18.39  zero_weeks= 0/23  nonzero_weeks=23  -> OK
[M4c]   CBD 1                     avg/wk=  1.13  zero_weeks=12/23  nonzero_weeks=11  -> LOW_CONFIDENCE (avg_weekly=1.13 < 8.0; zero_week_fraction=0.52 > 0.25)
[M4c]   CBD 2                     avg/wk=

In [ ]:
!python /content/drive/MyDrive/flipkart/m5_inference_engine.py

[M5] Loaded M4c forecast context.
[M5] Loaded artifacts from /content/drive/MyDrive/flipkart/
[M5] Closure model: blend(ExtraTrees_w6 x0.5 + RandomForest_w4 x0.5)
[M5] CIS scorer model class: XGBRegressor

[M5] DEMO EVENT 1
{
  "input": {
    "latitude": 12.9716,
    "longitude": 77.5573,
    "event_cause": "vehicle_breakdown",
    "event_type": "unplanned",
    "priority": "LOW",
    "timestamp": "2026-06-20T17:30:00+00:00",
    "was_escalated": false,
    "description": "lorry breakdown on left lane, slow traffic",
    "address": "Mysore Road, Bengaluru"
  },
  "resolved_location": {
    "corridor_final": "Non-corridor",
    "corridor_confidence": 0.74,
    "corridor_agreement": 0,
    "corridor_source": "unresolved",
    "rf_corridor_prediction": "Mysore Road",
    "rf_corridor_confidence": 0.48,
    "knn_corridor_prediction": "Magadi Road",
    "knn_corridor_confidence": 1.0,
    "police_station": "Magadi Road",
    "police_station_nn_distance_km": 0.45525158518775516
  },
  "basel

In [ ]:
!python /content/drive/MyDrive/flipkart/m6_resource_rule_engine.py

[M5] Loaded M4c forecast context.
[M5] Loaded artifacts from /content/drive/MyDrive/flipkart/
[M5] Closure model: blend(ExtraTrees_w6 x0.5 + RandomForest_w4 x0.5)
[M5] CIS scorer model class: XGBRegressor

[M6] DEMO EVENT 1
{
  "cis_used": 2.289,
  "cis_used_source": "cis_ml_based",
  "cis_based_tier": "LOW",
  "closure_predicted": false,
  "closure_probability": 0.1874,
  "closure_override_applied": false,
  "final_tier": "LOW",
  "officer_count": 1,
  "barricade_count": 0,
  "diversion_priority": "NONE",
  "rationale": "Minor incident, bottom quartile of the ML-predicted CIS distribution, historical closure rate in this band is ~5.3%. Single officer for traffic guidance/logging; no physical barricading by default.",
  "cis_disagreement_check": {
    "checked": true,
    "cis_formula_based": 2.544,
    "cis_ml_based": 2.289,
    "absolute_gap": 0.255,
    "flagged_as_high_disagreement": false
  },
  "tier_thresholds_used": [
    {
      "name": "LOW",
      "min_cis": 0.0,
      "max_

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/flipkart/static", exist_ok=True)
print("Done")

Done


In [10]:
from pyngrok import ngrok
ngrok.kill()

ngrok.set_auth_token("Enter your ngrok key")
tunnel = ngrok.connect(8501, domain="implant-outdated-saggy.ngrok-free.dev")
print("Streamlit URL:", tunnel.public_url)

Streamlit URL: https://implant-outdated-saggy.ngrok-free.dev


In [ ]:
!streamlit run /content/drive/MyDrive/flipkart/streamlit_app.py \
  --server.port 8501 \
  --server.enableStaticServing true



<unknown>:808: SyntaxWarning: invalid escape sequence '\s'
2026-06-23 17:26:50.278 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.228.5.164:8501

/content/drive/MyDrive/flipkart/streamlit_app.py:808: SyntaxWarning: invalid escape sequence '\s'
  if not mappls_key:
2026-06-23 17:27:24.858 Please replace `st.components.v1.iframe` with `st.iframe`.

`st.components.v1.iframe` will be removed after 2026-06-01.
2026-06-23 17:27:25.135 Please replace `st.components.v1.iframe` with `st.iframe`.

`st.components.v1.iframe` will be removed after 2026-06-01.
2026-06-23 17:27:35.456 Please replace `st.components.v1.iframe` with `st.iframe`.

`st.components.v1.iframe` will be removed after 2026-06-01.
2026-06-23 17:27:44.612 Please replace `st.components.v1.iframe` with `st.iframe`.

`st.components.v1.iframe` will be removed after 2026-06-01.
[M5] Load

In [ ]:
!python3 corridor_reconstruction.py

[RECON] corridor_reconstruction.py  v2 — AUTHENTIC ROAD NAMES
[RECON] Started  2026-06-22 07:35:21
[RECON] Mode: storing real API road names — NO corridor matching

[RECON] Loaded CSV   : 8173 total rows, 46 columns
[RECON] Named corridors (UNTOUCHED)  : 5049
[RECON] Non-corridor  (to process)   : 3124

[RECON]    50/3124  updated=50  errors=0  cache=0 (0%)
[RECON]   100/3124  updated=100  errors=0  cache=1 (1%)
[RECON]   150/3124  updated=150  errors=0  cache=1 (0%)
[RECON]   200/3124  updated=200  errors=0  cache=2 (1%)
[RECON]   250/3124  updated=250  errors=0  cache=2 (0%)
[RECON]   300/3124  updated=300  errors=0  cache=2 (0%)
[RECON]   350/3124  updated=350  errors=0  cache=4 (1%)
[RECON]   400/3124  updated=400  errors=0  cache=4 (1%)
[RECON]   450/3124  updated=450  errors=0  cache=4 (0%)
[RECON]   500/3124  updated=500  errors=0  cache=5 (1%)
[RECON]   550/3124  updated=550  errors=0  cache=6 (1%)
[RECON]   600/3124  updated=600  errors=0  cache=6 (1%)
[RECON]   650/3124  upda